In this notebook, we perform the analyses pertaining to figure 5. 
This includes 
- registration of data to the templates used in the study
- masking and translation of mouse Alzheimer's map, comparison with human map (permutation test)
- creation and translation of mouse Parkinson's map, comparison with human map (permutation test)

In [ ]:
# Imports
# Imports
import pandas as pd
import numpy as np
import os

import nibabel as nib
import nilearn
from nilearn import plotting

import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import neuromaps
from neuromaps import nulls, transforms, resampling
from neuromaps import stats as nm_stats
from neuromaps.datasets import fetch_annotation

from nilearn import surface, datasets

In [ ]:
# Function definitions

def mirror_brain(half_map):
    human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
    affine = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').affine
    
    human_map = human_brain.copy()
    human_half_map = human_map[0:50,:,:]
    human_half_map[human_half_map>0] = half_map
    human_map[0:50,:,:]=human_half_map
    human_map[50:,:,:]=np.flip(human_half_map[1:49,:,:], axis=0)
    
    map_ = nib.Nifti1Image(human_map, affine)
    return map_

def translate_mouse_human(mouse_roi):
    human_labels = pd.read_csv('data/mouse_human/data.ign/atlas_ahba.csv')
    long_label = human_labels['Long label']
    
    human_brain = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').get_fdata()
    affine = nib.load('data/mouse_human/data.ign/full_atlas_129_regions_resampled_2mm.nii.gz').affine

    mouse_vox_embedding = pd.read_csv('/well/mars/users/uvy786/dl_mouse_human/exps/e3_vae/results.ign/sweep_result/encoding/mouse_voxel_encoding_with_scaling.csv')
    mouse_regions = mouse_vox_embedding.iloc[:,-1]

    similarity=np.load('similarity_scored.csv.npy')

    temp = [similarity[mouse_regions==m_roi,:] for m_roi in mouse_roi]
    translated_half_map = []
    for i in range(len(temp)):
        translated_half_map += [*temp[i]]

    translated_half_map = np.sum(translated_half_map, axis=0)
    
    scaled = (translated_half_map - translated_half_map.min()) / (translated_half_map.max() - translated_half_map.min())
    unthresholded_translated = scaled.copy()
    
    unthresholded=mirror_brain(unthresholded_translated)

    threshold=0.6 

    whole_map=unthresholded.get_fdata()
    whole_map[whole_map<threshold]=0
    whole_map = whole_map.reshape(unthresholded.get_fdata().shape)
    whole_map = nib.Nifti1Image(whole_map, unthresholded.affine)

    return whole_map, unthresholded, threshold


def plot_human_surf(mask_nii_h):
    fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage5")
    hemi= "left"
    radius = 8
    kind = 'ball'
    depth = 0.7
    inter='linear'
    n_samples=160
    pial_mesh = fsaverage[f"pial_{hemi}"]
    infl_mesh = fsaverage[f"infl_{hemi}"]
    X = surface.vol_to_surf(mask_nii_h, pial_mesh, radius=radius, interpolation=inter, n_samples=n_samples, kind=kind).T
    X = (X - X.min()) /(X.max() - X.min())
    nilearn.plotting.plot_surf_stat_map(infl_mesh,X,view=("lateral"), cmap='jet', colorbar=True)
    nilearn.plotting.plot_surf_stat_map(infl_mesh,X, view=("medial"), cmap='jet')

In [ ]:
# Data

In [ ]:
# AD

# Mouse data

# Human data

# Translate mouse

# Compare translation and real human data 


In [ ]:
# PD
# Make mouse masks based on the MPTP mouse model effects

# 